# Word2Vec using Gensim

In [1]:
"""
low_resource_nlp_toolkit.py
============================
Single-file generic NLP evaluation toolkit for morphologically rich languages.
Works in Jupyter notebooks, Google Colab, or as a standalone script.

Usage in Jupyter:
    from low_resource_nlp_toolkit import LowResourceEvaluator, get_language, list_languages

    # List available languages
    list_languages()

    # Run evaluation
    ev = LowResourceEvaluator(get_language("zul"))
    ev.run()

    # Compare languages
    compare_languages(["zul", "xho", "swa"])
"""

# ── Auto-install ──────────────────────────────────────────────────────────────
import subprocess, sys

def _ensure(libs):
    for lib in libs:
        try:
            __import__(lib)
        except ImportError:
            print(f"📦 Installing {lib}…")
            subprocess.check_call([sys.executable, "-m", "pip", "install", lib, "-q",
                                   "--break-system-packages"], stderr=subprocess.DEVNULL)

_ensure(["numpy", "scipy", "gensim", "nltk"])

# ── Standard imports ──────────────────────────────────────────────────────────
import os, csv, time
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

import numpy as np
from scipy.stats import spearmanr, pearsonr
from gensim.models import FastText


# ══════════════════════════════════════════════════════════════════════════════
# 1.  LanguageConfig  — everything language-specific lives here
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class LanguageConfig:
    """
    Inherit and override fields to add a new language.
    The evaluator reads every setting from this object — no hardcoding elsewhere.
    """

    # Identity
    language_code:   str = "und"
    language_name:   str = "Unknown"
    language_family: str = "Unknown"

    # Morphological flags
    is_agglutinative:   bool = True
    is_tonal:           bool = False
    preserve_case:      bool = True    # True → no lowercasing (noun-class prefixes)
    preserve_diacritics:bool = True

    # FastText subword n-gram range
    # Bantu/Turkic → 3-6 | Dravidian → 2-5 | Isolating → 1-3
    min_n: int = 3
    max_n: int = 6

    # Model hyperparameters
    vector_size: int   = 200
    window:      int   = 7
    min_count:   int   = 2
    epochs:      int   = 100
    sg:          int   = 1        # 1=skip-gram (better for small corpora)
    alpha:       float = 0.025
    min_alpha:   float = 0.0001
    negative:    int   = 10
    sample:      float = 1e-1
    workers:     int   = 4
    bucket:      int   = 2_000_000

    similarity_threshold: float = 0.3

    # Auto-derived filenames (override if needed)
    corpus_file:  str = ""
    output_csv:   str = ""
    metrics_file: str = ""

    # Language data
    test_pairs:                List[Tuple[str, str, float]] = field(default_factory=list)
    sample_corpus_sentences:   List[str]                   = field(default_factory=list)
    oov_demo_words:            List[str]                   = field(default_factory=list)
    translation_map:           Dict[str, str]              = field(default_factory=dict)

    def __post_init__(self):
        if not self.corpus_file:
            self.corpus_file  = f"{self.language_code}_corpus.txt"
        if not self.output_csv:
            self.output_csv   = f"{self.language_code}_fasttext_results.csv"
        if not self.metrics_file:
            self.metrics_file = f"{self.language_code}_evaluation_metrics.txt"

    def description(self) -> str:
        flags = [f for f, v in [
            ("agglutinative", self.is_agglutinative),
            ("tonal",         self.is_tonal),
            ("case-sensitive",self.preserve_case),
        ] if v]
        return f"{self.language_name} [{self.language_code}] ({self.language_family}) — {', '.join(flags)}"


# ══════════════════════════════════════════════════════════════════════════════
# 2.  Built-in language configurations
# ══════════════════════════════════════════════════════════════════════════════

# ── isiZulu ───────────────────────────────────────────────────────────────────
ISIZULU = LanguageConfig(
    language_code="zul", language_name="isiZulu", language_family="Bantu / Nguni",
    is_agglutinative=True, is_tonal=True, preserve_case=True, preserve_diacritics=True,
    min_n=3, max_n=6, vector_size=200, window=7, epochs=100, min_count=2,
    sample_corpus_sentences=[
        "umfazi nendoda bahamba esikoleni",
        "ingane idla ukudla kwayo",
        "inja ikati zidlala eyadini",
        "isikole isikhungo semfundo",
        "ikhaya indlu yomndeni",
        "umfula ulwandle amanzi",
        "uthisha umfundi bafunda",
        "ibhola umdlalo imidlalo",
        "inkosi indlovukazi umbuso",
        "imali ibhange ukonga",
        "udokotela umhlengikazi ukwelapha",
        "ikhompiyutha ikhibhodi theknoloji",
        "indiza imoto isitimela ukuhamba",
        "inyoni iqhude izilwane",
        "isinkwa ibhotela ukudla",
        "imeya inkosi amadolobha",
        "isikweletu imali ukuboleka",
        "umasipala uhulumeni isikhungo",
        "inkohlakalo icala ububi",
        "inyuvesi isikole ukufunda",
    ],
    test_pairs=[
        ("inkosi",      "imeya",        8.45),
        ("uhulumeni",   "umasipala",    8.90),
        ("isikole",     "inyuvesi",     8.20),
        ("umfundi",     "uthisha",      7.65),
        ("ingoma",      "icwecwe",      8.55),
        ("ubaba",       "umama",        8.90),
        ("umndeni",     "ikhaya",       8.40),
        ("inja",        "ikati",        7.90),
        ("imali",       "isikweletu",   7.12),
        ("inkohlakalo", "icala",        7.50),
        ("ikhwaya",     "umbhalo",      4.10),
        ("idolobha",    "ilokishi",     7.30),
        ("itekisi",     "ubudokotela",  1.15),
        ("umculo",      "isifo",        0.90),
        ("inkosi",      "igwaba",       1.05),
        ("ibhubesi",    "isikweletu",   0.65),
    ],
    oov_demo_words=["ukuthenga", "abantwana", "izinkomo"],
    translation_map={
        "inkosi": "king",  "indlovukazi": "queen",
        "ubaba":  "father","umama":       "mother",
        "inja":   "dog",   "ikati":       "cat",
        "imali":  "money", "ibhange":     "bank",
        "isikole":"school","inyuvesi":    "university",
    },
)

# ── isiXhosa ──────────────────────────────────────────────────────────────────
ISIXHOSA = LanguageConfig(
    language_code="xho", language_name="isiXhosa", language_family="Bantu / Nguni",
    is_agglutinative=True, is_tonal=True, preserve_case=True, preserve_diacritics=True,
    min_n=3, max_n=6, vector_size=200, window=7, epochs=100, min_count=2,
    sample_corpus_sentences=[
        "umntu umfazi indoda bahamba esikolweni",
        "inja ikati badlala egadini",
        "isikolo isikhungo semfundo",
        "ikhaya indlu yomndeni",
        "umfula ulwandle amanzi",
        "utitshala umfundi bafunda",
        "ibhola umdlalo iindlela",
        "ukumkani indlovukazi umbuso",
        "imali ibhanki ukonga",
        "ugqirha umongikazi ukuphilisa",
        "ikhompyutha ikhiybhodi iteknoloji",
        "indiza imoto isitimela ukuhamba",
        "iinyoni izilwane",
        "isonka ibhotolo ukutya",
        "imayor ukumkani amadolophu",
    ],
    test_pairs=[
        ("ukumkani",  "imayor",      8.45),
        ("isikolo",   "yunivesithi", 8.20),
        ("utitshala", "umfundi",     7.65),
        ("inja",      "ikati",       7.90),
        ("umfazi",    "indoda",      7.80),
        ("imali",     "ibhanki",     7.60),
        ("ikhaya",    "indlu",       8.80),
        ("isonka",    "ukutya",      8.10),
        ("inja",      "isikolo",     1.20),
        ("imali",     "inyoni",      0.85),
        ("ugqirha",   "ibhola",      1.10),
    ],
    oov_demo_words=["ukulima", "abantwana", "izilwane"],
    translation_map={
        "ukumkani":   "king",   "indlovukazi": "queen",
        "inja":       "dog",    "ikati":       "cat",
        "isikolo":    "school", "yunivesithi": "university",
    },
)

# ── Kiswahili ─────────────────────────────────────────────────────────────────
SWAHILI = LanguageConfig(
    language_code="swa", language_name="Kiswahili", language_family="Bantu",
    is_agglutinative=True, is_tonal=False, preserve_case=False, preserve_diacritics=True,
    min_n=3, max_n=6, vector_size=200, window=7, epochs=100, min_count=2,
    sample_corpus_sentences=[
        "mwanafunzi mwalimu wanasoma shuleni",
        "mbwa na paka wanacheza nyumbani",
        "hospitali daktari wagonjwa",
        "mji mjini watu wengi",
        "chakula kinywaji maji",
        "serikali rais waziri mkuu",
        "fedha benki akiba",
        "ndege treni gari usafiri",
        "soko bei biashara",
        "familia baba mama watoto",
        "shule chuo kikuu elimu",
        "muziki wimbo msanii",
        "polisi mwizi uhalifu",
        "mvua mto bahari maji",
        "nyumba kijiji mji",
    ],
    test_pairs=[
        ("mwanafunzi","mwalimu",  7.65),
        ("shule",     "chuo",     8.20),
        ("daktari",   "hospitali",8.50),
        ("baba",      "mama",     8.90),
        ("mbwa",      "paka",     7.90),
        ("serikali",  "rais",     7.80),
        ("fedha",     "benki",    7.60),
        ("ndege",     "treni",    7.40),
        ("wimbo",     "muziki",   8.55),
        ("nyumba",    "familia",  7.50),
        ("mbwa",      "fedha",    0.80),
        ("ndege",     "shule",    1.10),
        ("muziki",    "daktari",  0.90),
    ],
    oov_demo_words=["wanafunzi", "mwalimu", "chakula"],
    translation_map={
        "mwalimu":    "teacher",    "mwanafunzi": "student",
        "daktari":    "doctor",     "hospitali":  "hospital",
        "mbwa":       "dog",        "paka":       "cat",
        "shule":      "school",     "chuo":       "university",
        "fedha":      "money",      "benki":      "bank",
    },
)

# ── Tamil ─────────────────────────────────────────────────────────────────────
TAMIL = LanguageConfig(
    language_code="tam", language_name="Tamil", language_family="Dravidian",
    is_agglutinative=True, is_tonal=False, preserve_case=False, preserve_diacritics=True,
    min_n=2, max_n=5, vector_size=200, window=5, epochs=100, min_count=2,
    sample_corpus_sentences=[
        "மாணவன் ஆசிரியர் பள்ளியில் படிக்கிறார்",
        "நாய் பூனை வீட்டில் விளையாடுகின்றன",
        "மருத்துவர் மருத்துவமனை நோயாளி",
        "நகரம் கிராமம் மக்கள்",
        "உணவு தண்ணீர் குடிக்கிறோம்",
        "அரசு அமைச்சர் நாடாளுமன்றம்",
        "பணம் வங்கி சேமிப்பு",
        "விமானம் ரயில் பேருந்து",
        "கடை விலை வணிகம்",
        "குடும்பம் அப்பா அம்மா குழந்தை",
    ],
    test_pairs=[
        ("மாணவன்",     "ஆசிரியர்",    7.65),
        ("பள்ளி",      "கல்லூரி",     8.20),
        ("மருத்துவர்", "மருத்துவமனை", 8.50),
        ("அப்பா",      "அம்மா",       8.90),
        ("நாய்",       "பூனை",        7.90),
        ("பணம்",       "வங்கி",       7.60),
        ("விமானம்",    "ரயில்",       7.40),
        ("உணவு",       "தண்ணீர்",     6.80),
        ("நாய்",       "பணம்",        0.80),
        ("விமானம்",    "பள்ளி",       1.10),
    ],
    oov_demo_words=["மாணவர்கள்", "மருத்துவர்கள்", "குழந்தைகள்"],
    translation_map={
        "மாணவன்": "student", "ஆசிரியர்": "teacher",
        "நாய்":   "dog",     "பூனை":     "cat",
        "பணம்":   "money",   "வங்கி":    "bank",
    },
)

# ── Turkish ───────────────────────────────────────────────────────────────────
TURKISH = LanguageConfig(
    language_code="tur", language_name="Turkish", language_family="Turkic",
    is_agglutinative=True, is_tonal=False, preserve_case=False, preserve_diacritics=True,
    min_n=3, max_n=6, vector_size=200, window=7, epochs=100, min_count=2,
    sample_corpus_sentences=[
        "öğrenci öğretmen okulda okuyor",
        "köpek kedi evde oynuyor",
        "doktor hastane hasta",
        "şehir köy insanlar",
        "yemek içecek su",
        "hükümet bakan meclis",
        "para banka tasarruf",
        "uçak tren otobüs",
        "market fiyat ticaret",
        "aile baba anne çocuk",
        "müzik şarkı sanatçı",
        "polis suç hırsız",
        "yağmur nehir deniz su",
        "ev mahalle şehir",
        "üniversite fakülte öğrenci",
    ],
    test_pairs=[
        ("öğrenci",  "öğretmen",   7.65),
        ("okul",     "üniversite", 8.20),
        ("doktor",   "hastane",    8.50),
        ("baba",     "anne",       8.90),
        ("köpek",    "kedi",       7.90),
        ("hükümet",  "bakan",      7.80),
        ("para",     "banka",      7.60),
        ("uçak",     "tren",       7.40),
        ("şarkı",    "müzik",      8.55),
        ("ev",       "aile",       7.50),
        ("köpek",    "para",       0.80),
        ("uçak",     "okul",       1.10),
        ("müzik",    "doktor",     0.90),
    ],
    oov_demo_words=["öğrenciler", "doktorlar", "çocuklar"],
    translation_map={
        "öğretmen":  "teacher",    "öğrenci":  "student",
        "doktor":    "doctor",     "hastane":  "hospital",
        "köpek":     "dog",        "kedi":     "cat",
        "okul":      "school",     "üniversite":"university",
        "para":      "money",      "banka":    "bank",
    },
)

# ── Registry ──────────────────────────────────────────────────────────────────
LANGUAGE_REGISTRY: Dict[str, LanguageConfig] = {
    "zul": ISIZULU,
    "xho": ISIXHOSA,
    "swa": SWAHILI,
    "tam": TAMIL,
    "tur": TURKISH,
}

def get_language(code: str) -> LanguageConfig:
    code = code.lower()
    if code not in LANGUAGE_REGISTRY:
        available = ", ".join(LANGUAGE_REGISTRY.keys())
        raise ValueError(f"Language '{code}' not registered. Available: {available}")
    return LANGUAGE_REGISTRY[code]

def list_languages() -> None:
    print("\n📋 Registered languages:")
    for code, cfg in LANGUAGE_REGISTRY.items():
        print(f"   [{code}]  {cfg.description()}  —  {len(cfg.test_pairs)} test pairs")
    print()


# ══════════════════════════════════════════════════════════════════════════════
# 3.  Maths / metrics helpers
# ══════════════════════════════════════════════════════════════════════════════

def cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    return float(np.dot(v1, v2) / (n1 * n2)) if (n1 and n2) else 0.0

def _confusion(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return tp, tn, fp, fn

def _clf_metrics(tp, tn, fp, fn):
    total = tp + tn + fp + fn
    acc   = (tp + tn) / total         if total        else 0.0
    prec  = tp / (tp + fp)            if (tp + fp)    else 0.0
    rec   = tp / (tp + fn)            if (tp + fn)    else 0.0
    f1    = 2*prec*rec/(prec+rec)     if (prec+rec)   else 0.0
    return acc, prec, rec, f1


# ══════════════════════════════════════════════════════════════════════════════
# 4.  WordNet cross-lingual validation (optional)
# ══════════════════════════════════════════════════════════════════════════════

def wordnet_validation(model_wv, translation_map: Dict[str, str], top_n: int = 5) -> None:
    try:
        import nltk
        from nltk.corpus import wordnet as wn
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
    except Exception:
        print("   ⚠️  NLTK WordNet not available — skipping.")
        return

    print("\n📚 WordNet Cross-Lingual Validation")
    print("─" * 60)
    found = 0
    for word_l2, word_en in translation_map.items():
        try:
            neighbours = model_wv.most_similar(word_l2, topn=top_n)
        except KeyError:
            continue
        for nbr, cos in neighbours:
            nbr_en = translation_map.get(nbr)
            if not nbr_en:
                continue
            s1 = wn.synsets(word_en)
            s2 = wn.synsets(nbr_en)
            if not s1 or not s2:
                continue
            sim = max((a.path_similarity(b) or 0.0) for a in s1 for b in s2)
            if sim > 0:
                print(f"   {word_l2}({word_en}) ↔ {nbr}({nbr_en}) "
                      f"cosine={cos:.4f}  WordNet={sim:.4f}")
                found += 1
    if not found:
        print("   (no overlapping translations found in WordNet)")
    print("─" * 60)


# ══════════════════════════════════════════════════════════════════════════════
# 5.  Core evaluator  (language-agnostic)
# ══════════════════════════════════════════════════════════════════════════════

class LowResourceEvaluator:
    """
    Full pipeline: corpus → FastText training → evaluation → report → save.

    Parameters
    ----------
    config : LanguageConfig
        Any LanguageConfig instance — built-in or custom.

    Quick start
    -----------
    >>> ev = LowResourceEvaluator(get_language("zul"))
    >>> results = ev.run()
    """

    def __init__(self, config: LanguageConfig):
        self.cfg   = config
        self.model = None

    # ── Corpus ────────────────────────────────────────────────────────────────

    def prepare_corpus(self) -> List[List[str]]:
        cfg = self.cfg
        if not os.path.exists(cfg.corpus_file):
            if not cfg.sample_corpus_sentences:
                raise FileNotFoundError(
                    f"No corpus at '{cfg.corpus_file}' and no sample sentences in config."
                )
            print(f"   ℹ️  Writing built-in sample corpus → {cfg.corpus_file}")
            with open(cfg.corpus_file, "w", encoding="utf-8") as f:
                f.write("\n".join(cfg.sample_corpus_sentences) + "\n")

        sentences = []
        with open(cfg.corpus_file, "r", encoding="utf-8") as f:
            for line in f:
                text   = line.strip() if cfg.preserve_case else line.strip().lower()
                tokens = text.split()
                if tokens:
                    sentences.append(tokens)

        print(f"   ✅ Corpus: {len(sentences)} sentences from '{cfg.corpus_file}'")
        return sentences

    # ── Training ──────────────────────────────────────────────────────────────

    def train(self, sentences: List[List[str]]) -> None:
        cfg = self.cfg
        print(f"\n🚀 Training FastText — {cfg.language_name}")
        print(f"   n-grams={cfg.min_n}–{cfg.max_n}  dim={cfg.vector_size}  "
              f"window={cfg.window}  epochs={cfg.epochs}  min_count={cfg.min_count}")
        t0 = time.time()
        self.model = FastText(
            sentences=sentences,
            vector_size=cfg.vector_size,
            window=cfg.window,
            min_count=cfg.min_count,
            epochs=cfg.epochs,
            sg=cfg.sg,
            workers=cfg.workers,
            alpha=cfg.alpha,
            min_alpha=cfg.min_alpha,
            negative=cfg.negative,
            sample=cfg.sample,
            min_n=cfg.min_n,
            max_n=cfg.max_n,
            word_ngrams=1,
            bucket=cfg.bucket,
        )
        print(f"   ✅ Done in {time.time()-t0:.1f}s  —  vocab={len(self.model.wv)} tokens")

    # ── Similarity table ──────────────────────────────────────────────────────

    def compute_similarities(self) -> Tuple[List[float], List[float], List[dict]]:
        wv = self.model.wv
        human_scores, cosine_scores, rows = [], [], []

        print(f"\n{'Word 1':<25} {'Word 2':<25} {'Human':>7} {'Cosine':>9} {'Status'}")
        print("─" * 80)

        for w1, w2, human in self.cfg.test_pairs:
            try:
                cos    = cosine_similarity(wv[w1], wv[w2])
                in1    = w1 in wv.key_to_index
                in2    = w2 in wv.key_to_index
                status = "InVocab" if (in1 and in2) else ("Partial" if (in1 or in2) else "N-gram")
                print(f"{w1:<25} {w2:<25} {human:>7.2f} {cos:>9.6f} {status}")
                human_scores.append(human)
                cosine_scores.append(cos)
                rows.append({"word1": w1, "word2": w2,
                             "human_score": human, "cosine_similarity": cos,
                             "vocab_status": status})
            except Exception as e:
                print(f"{w1:<25} {w2:<25} {human:>7.2f}   ERROR: {e}")

        print("─" * 80)
        return human_scores, cosine_scores, rows

    # ── Metrics ───────────────────────────────────────────────────────────────

    def compute_metrics(self, human: List[float], cosine: List[float]) -> dict:
        rho,  rho_p  = spearmanr(human, cosine)
        pear, pear_p = pearsonr(human, cosine)

        h_med  = float(np.median(human))
        c_med  = float(np.median(cosine))
        y_true = (np.array(human)  >= h_med).astype(int)
        y_pred = (np.array(cosine) >= c_med).astype(int)

        tp, tn, fp, fn   = _confusion(y_true, y_pred)
        acc, prec, rec, f1 = _clf_metrics(tp, tn, fp, fn)

        return dict(
            spearman=rho,   spearman_p=rho_p,
            pearson=pear,   pearson_p=pear_p,
            accuracy=acc,   precision=prec, recall=rec, f1=f1,
            tp=tp, tn=tn, fp=fp, fn=fn,
            n_pairs=len(human),
            cos_min=min(cosine), cos_max=max(cosine),
            cos_mean=float(np.mean(cosine)), cos_std=float(np.std(cosine)),
            h_min=min(human),   h_max=max(human),
            h_mean=float(np.mean(human)),    h_std=float(np.std(human)),
        )

    # ── Report ────────────────────────────────────────────────────────────────

    def print_report(self, m: dict) -> None:
        cfg  = self.cfg
        line = "═" * 65

        def label(rho):
            return ("EXCELLENT ✅" if rho >= .70 else
                    "GOOD 🟡"      if rho >= .50 else
                    "FAIR 🟠"      if rho >= .30 else "POOR 🔴")

        print(f"\n{line}")
        print(f"  RESULTS — {cfg.language_name}  ({cfg.language_code})")
        print(f"  {cfg.description()}")
        print(line)
        print(f"\n  📊 CORRELATION")
        print(f"     Spearman ρ : {m['spearman']:+.6f}  p={m['spearman_p']:.6f}  → {label(abs(m['spearman']))}")
        print(f"     Pearson  r : {m['pearson']:+.6f}  p={m['pearson_p']:.6f}")
        print(f"\n  📈 CLASSIFICATION  (binarised at median cosine)")
        print(f"     Accuracy   : {m['accuracy']:.4f}")
        print(f"     Precision  : {m['precision']:.4f}")
        print(f"     Recall     : {m['recall']:.4f}")
        print(f"     F1-Score   : {m['f1']:.4f}")
        print(f"\n  🔢 CONFUSION MATRIX")
        print(f"     TP={m['tp']}  FP={m['fp']}  FN={m['fn']}  TN={m['tn']}")
        print(f"\n  📉 COSINE  (n={m['n_pairs']} pairs)")
        print(f"     min={m['cos_min']:.4f}  max={m['cos_max']:.4f}  "
              f"mean={m['cos_mean']:.4f}  std={m['cos_std']:.4f}")
        print(f"\n  🔤 SUBWORD CONFIG")
        print(f"     n-gram range    : {cfg.min_n}–{cfg.max_n} chars")
        print(f"     Agglutinative   : {cfg.is_agglutinative}  "
              f"Tonal: {cfg.is_tonal}  "
              f"Preserve case: {cfg.preserve_case}")
        print(line + "\n")

    # ── OOV demo ──────────────────────────────────────────────────────────────

    def demo_oov(self) -> None:
        if not self.cfg.oov_demo_words:
            return
        print("🔬 OOV word handling (n-gram fallback)")
        print("─" * 45)
        for w in self.cfg.oov_demo_words:
            try:
                self.model.wv[w]
                status = "In vocab" if w in self.model.wv.key_to_index else "From n-grams"
                print(f"   '{w}'  →  {status} ✓")
            except Exception:
                print(f"   '{w}'  →  FAILED ✗")
        print("─" * 45)

    # ── Save ──────────────────────────────────────────────────────────────────

    def save_outputs(self, rows: List[dict], m: dict) -> None:
        cfg = self.cfg
        with open(cfg.output_csv, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"   💾 Pair CSV    → {cfg.output_csv}")

        with open(cfg.metrics_file, "w", encoding="utf-8") as f:
            f.write(f"Low-Resource NLP — {cfg.language_name}\n{'='*60}\n\n")
            f.write(f"Language family : {cfg.language_family}\n")
            f.write(f"ISO code        : {cfg.language_code}\n")
            f.write(f"n-gram range    : {cfg.min_n}–{cfg.max_n}\n\n")
            f.write(f"CORRELATION\n"
                    f"  Spearman ρ : {m['spearman']:.6f} (p={m['spearman_p']:.6f})\n"
                    f"  Pearson  r : {m['pearson']:.6f}  (p={m['pearson_p']:.6f})\n\n")
            f.write(f"CLASSIFICATION\n"
                    f"  Accuracy   : {m['accuracy']:.6f}\n"
                    f"  Precision  : {m['precision']:.6f}\n"
                    f"  Recall     : {m['recall']:.6f}\n"
                    f"  F1-Score   : {m['f1']:.6f}\n\n"
                    f"  TP={m['tp']}  FP={m['fp']}  FN={m['fn']}  TN={m['tn']}\n\n")
            f.write(f"COVERAGE\n"
                    f"  Pairs evaluated: {m['n_pairs']}/{len(cfg.test_pairs)}\n")
        print(f"   💾 Metrics txt → {cfg.metrics_file}")

    # ── Full pipeline ─────────────────────────────────────────────────────────

    def run(self, run_wordnet: bool = True, external_corpus: Optional[str] = None) -> dict:
        """
        Run the complete pipeline and return a metrics dict.

        Parameters
        ----------
        run_wordnet      : attempt WordNet cross-lingual validation
        external_corpus  : path to your own corpus file (overrides config)
        """
        cfg = self.cfg
        if external_corpus:
            cfg.corpus_file = external_corpus

        print(f"\n{'═'*65}")
        print(f"  Low-Resource NLP Pipeline  —  {cfg.language_name}")
        print(f"{'═'*65}\n")

        sentences                        = self.prepare_corpus()
        self.train(sentences)
        human, cosine, rows              = self.compute_similarities()

        if len(cosine) < 2:
            print("❌ Fewer than 2 valid pairs — cannot compute metrics.")
            return {}

        metrics = self.compute_metrics(human, cosine)
        self.print_report(metrics)
        self.demo_oov()

        if run_wordnet and cfg.translation_map:
            wordnet_validation(self.model.wv, cfg.translation_map)

        self.save_outputs(rows, metrics)
        return metrics


# ══════════════════════════════════════════════════════════════════════════════
# 6.  Multi-language comparison helper
# ══════════════════════════════════════════════════════════════════════════════

def compare_languages(codes: List[str], run_wordnet: bool = False) -> None:
    """
    Evaluate multiple languages and print a side-by-side comparison table.

    Parameters
    ----------
    codes       : list of ISO 639-3 codes, e.g. ["zul", "xho", "swa"]
    run_wordnet : include WordNet validation for each language
    """
    results = {}
    for code in codes:
        cfg = get_language(code)
        ev  = LowResourceEvaluator(cfg)
        m   = ev.run(run_wordnet=run_wordnet)
        if m:
            results[code] = (cfg.language_name, m)

    if len(results) < 2:
        return

    W = 12
    print("\n" + "═" * 80)
    print("  CROSS-LANGUAGE COMPARISON")
    print("═" * 80)
    print(f"  {'Language':<18} {'Code':>5} {'Spearman':>{W}} {'Pearson':>{W}} "
          f"{'Accuracy':>{W}} {'F1':>{W}} {'Pairs':>{W}}")
    print("  " + "─" * 78)
    for code, (name, m) in results.items():
        print(f"  {name:<18} {code:>5} "
              f"{m['spearman']:>{W}.4f} {m['pearson']:>{W}.4f} "
              f"{m['accuracy']:>{W}.4f} {m['f1']:>{W}.4f} {m['n_pairs']:>{W}}")
    print("═" * 80)

    best = max(results, key=lambda c: abs(results[c][1]["spearman"]))
    print(f"\n  🏆 Best Spearman: {results[best][0]} ({best})  "
          f"ρ={results[best][1]['spearman']:.4f}\n")


# ══════════════════════════════════════════════════════════════════════════════
# 7.  Template for adding a new language — copy and fill in
# ══════════════════════════════════════════════════════════════════════════════

NEW_LANGUAGE_TEMPLATE = """
# Copy this block, fill in every value, then add to LANGUAGE_REGISTRY:
#   LANGUAGE_REGISTRY["???"] = MY_NEW_LANG

MY_NEW_LANG = LanguageConfig(
    language_code   = "???",           # ISO 639-3
    language_name   = "MyLanguage",
    language_family = "MyFamily",

    # Morphology flags
    is_agglutinative   = True,
    is_tonal           = False,
    preserve_case      = False,
    preserve_diacritics= True,

    # n-gram range (Bantu/Turkic=3-6, Dravidian=2-5, Isolating=1-3)
    min_n = 3,
    max_n = 6,

    # Model (safe defaults for < 5k sentences)
    vector_size=200, window=7, epochs=100, min_count=2,

    sample_corpus_sentences=[
        "sentence with related words together",  # >= 20 lines recommended
    ],

    test_pairs=[
        ("synonym_a",  "synonym_b",  9.0),   # high
        ("related_a",  "related_b",  6.5),   # medium
        ("unrelated1", "unrelated2", 0.9),   # negative — include several!
    ],

    oov_demo_words=["inflected_form", "compound_word"],

    translation_map={
        "native_word": "english_gloss",       # for WordNet validation
    },
)

LANGUAGE_REGISTRY["???"] = MY_NEW_LANG
"""


# ══════════════════════════════════════════════════════════════════════════════
# 8.  CLI (when run as a script)
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Generic Low-Resource NLP Evaluator")
    parser.add_argument("--lang",    metavar="CODE", help="Single language ISO code")
    parser.add_argument("--compare", metavar="CODE", nargs="+", help="Compare languages")
    parser.add_argument("--list",    action="store_true", help="List registered languages")
    parser.add_argument("--corpus",  metavar="PATH", help="Custom corpus file path")
    parser.add_argument("--no-wordnet", action="store_true", help="Skip WordNet validation")
    args = parser.parse_args(args=[])

    if args.list:
        list_languages()
    elif args.compare:
        compare_languages(args.compare, run_wordnet=not args.no_wordnet)
    elif args.lang:
        ev = LowResourceEvaluator(get_language(args.lang))
        ev.run(run_wordnet=not args.no_wordnet, external_corpus=args.corpus)
    else:
        list_languages()
        print("Run with --lang <code> or --compare <codes...>\n")
        print("Quick demo — running isiZulu:")
        ev = LowResourceEvaluator(ISIZULU)
        ev.run(run_wordnet=False)


📋 Registered languages:
   [zul]  isiZulu [zul] (Bantu / Nguni) — agglutinative, tonal, case-sensitive  —  16 test pairs
   [xho]  isiXhosa [xho] (Bantu / Nguni) — agglutinative, tonal, case-sensitive  —  11 test pairs
   [swa]  Kiswahili [swa] (Bantu) — agglutinative  —  13 test pairs
   [tam]  Tamil [tam] (Dravidian) — agglutinative  —  10 test pairs
   [tur]  Turkish [tur] (Turkic) — agglutinative  —  13 test pairs

Run with --lang <code> or --compare <codes...>

Quick demo — running isiZulu:

═════════════════════════════════════════════════════════════════
  Low-Resource NLP Pipeline  —  isiZulu
═════════════════════════════════════════════════════════════════

   ✅ Corpus: 1184 sentences from 'zul_corpus.txt'

🚀 Training FastText — isiZulu
   n-grams=3–6  dim=200  window=7  epochs=100  min_count=2
   ✅ Done in 10.9s  —  vocab=4001 tokens

Word 1                    Word 2                      Human    Cosine Status
─────────────────────────────────────────────────────────────────